# 01 — Data Collection
Pulling historical home game attendance data for the San Diego Padres from the MLB Stats API.

Note: `requests` is a Python library that lets you fetch data from the internet in your code — basically the same thing as typing a URL in your browser, but done programmatically (through code).

In [37]:
import pandas as pd
import numpy as np
import requests

print(f"pandas {pd.__version__} ✓")
print(f"numpy {np.__version__} ✓")
print("requests ✓")

pandas 3.0.3 ✓
numpy 2.4.6 ✓
requests ✓


## Pull Home Game Data
The MLB Stats API is free and official. We pull every regular season home game for the Padres across 5 seasons (2022-2025, post-COVID).

San Diego Padres team ID = 135.

Note: `game_pk` is the unique game ID assigned by MLB. PK stands for 'primary key' — a database term for a unique identifier. We'll use it later to link tables together.

In [23]:
TEAM_ID = 135  # San Diego Padres
SEASONS = [2022, 2023, 2024, 2025]

def get_home_games(team_id, season):
    url = "https://statsapi.mlb.com/api/v1/schedule"
    params = {
        "sportId": 1,
        "season": season,
        "gameType": "R",
        "teamId": team_id,
        "hydrate": "linescore,team",
    }
    response = requests.get(url, params=params)
    data = response.json()

    games = []
    for date in data.get("dates", []):
        for game in date.get("games", []):
            home_team_id = game["teams"]["home"]["team"]["id"]
            if home_team_id == team_id:
                # Try to get attendance — it lives at the top level of each game object
                attendance = game.get("attendance") or game.get("teams", {}).get("home", {}).get("attendance")
                games.append({
                    "date": date["date"],
                    "season": season,
                    "opponent": game["teams"]["away"]["team"]["name"],
                    "home_score": game["teams"]["home"].get("score"),
                    "away_score": game["teams"]["away"].get("score"),
                    "attendance": attendance,
                    "status": game["status"]["detailedState"],
                    "game_pk": game["gamePk"]
                })
    return games

# Pull all seasons
all_games = []
for season in SEASONS:
    games = get_home_games(TEAM_ID, season)
    all_games.extend(games)
    print(f"{season}: {len(games)} home games found")

df = pd.DataFrame(all_games)
print(f"\nTotal: {len(df)} games")
print(f"Games with attendance data: {df['attendance'].notna().sum()}")
print(f"Games missing attendance: {df['attendance'].isna().sum()}")
df.head(10)

2022: 81 home games found
2023: 81 home games found
2024: 81 home games found
2025: 81 home games found

Total: 324 games
Games with attendance data: 0
Games missing attendance: 324


,date,season,opponent,home_score,away_score,attendance,status,game_pk
0,2022-04-14,2022,Atlanta Braves,12,1,None,Final,662237
1,2022-04-15,2022,Atlanta Braves,2,5,None,Final,662236
2,2022-04-16,2022,Atlanta Braves,2,5,None,Final,662235
3,2022-04-17,2022,Atlanta Braves,2,1,None,Final,662234
4,2022-04-18,2022,Cincinnati Reds,4,1,None,Final,662233
5,2022-04-19,2022,Cincinnati Reds,6,2,None,Final,662232
6,2022-04-20,2022,Cincinnati Reds,6,0,None,Final,662231
7,2022-04-22,2022,Los Angeles Dodgers,1,6,None,Final,662230
8,2022-04-23,2022,Los Angeles Dodgers,3,2,None,Final,662276
9,2022-04-24,2022,Los Angeles Dodgers,2,10,None,Final,662274


## Fetch Attendance via Individual Game Endpoints
If attendance is still showing as None above, we fetch it game-by-game using each game's unique game_pk.
This is slower but guaranteed to get the official recorded attendance for every completed game.

In [24]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import re

def fetch_attendance(game_pk):
    try:
        url = "https://statsapi.mlb.com/api/v1/game/" + str(game_pk) + "/boxscore"
        data = requests.get(url, timeout=10).json()
        for item in data.get("info", []):
            if item.get("label") == "Att":
                val = re.sub(r"[^0-9]", "", item.get("value", ""))
                if val:
                    return game_pk, int(val)
    except:
        pass
    return game_pk, None

missing = df[df["attendance"].isna() & (df["status"] == "Final")]
print("Fetching attendance for", len(missing), "games in parallel...")

results = {}
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(fetch_attendance, pk): pk for pk in missing["game_pk"]}
    for future in as_completed(futures):
        game_pk, att = future.result()
        results[game_pk] = att

for idx, row in missing.iterrows():
    df.at[idx, "attendance"] = results.get(row["game_pk"])

filled = sum(1 for v in results.values() if v is not None)
print("Successfully fetched:", filled, "/", len(missing))
print("Sample attendance values:")
print(df[df["attendance"].notna()][["date", "opponent", "attendance"]].head(10))


Fetching attendance for 324 games in parallel...
Successfully fetched: 324 / 324
Sample attendance values:
         date             opponent attendance
0  2022-04-14       Atlanta Braves      44844
1  2022-04-15       Atlanta Braves      41993
2  2022-04-16       Atlanta Braves      36924
3  2022-04-17       Atlanta Braves      37694
4  2022-04-18      Cincinnati Reds      31121
5  2022-04-19      Cincinnati Reds      31313
6  2022-04-20      Cincinnati Reds      25359
7  2022-04-22  Los Angeles Dodgers      44482
8  2022-04-23  Los Angeles Dodgers      44444
9  2022-04-24  Los Angeles Dodgers      44930


## Save Training Data to CSV
Save 2022-2025 data to the data/ folder. This is our training data — we never modify this file.

Note - the stadium capacity of Petco Park is 42,445, but tickets sold for Gallagher Square (the Park in the Park - standing room only) accounts for values over capacity. The values are also for tickets sold, not actual attendees.

In [26]:
df.to_csv("../data/padres_games_raw.csv", index=False)
print("Saved to data/padres_games_raw.csv")
print(f"\nColumns: {list(df.columns)}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"\nAttendance summary:")
print(df['attendance'].astype(int).describe())

Saved to data/padres_games_raw.csv

Columns: ['date', 'season', 'opponent', 'home_score', 'away_score', 'attendance', 'status', 'game_pk']
Date range: 2022-04-14 to 2025-09-28

Attendance summary:
count      324.000000
mean     40206.080247
std       4863.651764
min      15952.000000
25%      37999.250000
50%      41787.000000
75%      43401.750000
max      47559.000000
Name: attendance, dtype: float64


## Pull 2026 Data (Prediction & Validation)
We keep 2026 completely separate from our training data.
- Games already played in 2026 let us check how accurate our model is against real results.
- Future 2026 games are genuine forecasts — no answer key yet.

In [30]:
games_2026 = get_home_games(TEAM_ID, 2026)
df_2026 = pd.DataFrame(games_2026)

# Fetch attendance for completed 2026 games using the same function as training data
missing_2026 = df_2026[df_2026["attendance"].isna() & (df_2026["status"] == "Final")]
print("Fetching attendance for", len(missing_2026), "completed 2026 games...")

results_2026 = {}
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(fetch_attendance, pk): pk for pk in missing_2026["game_pk"]}
    for future in as_completed(futures):
        game_pk, att = future.result()
        results_2026[game_pk] = att

for idx, row in missing_2026.iterrows():
    df_2026.at[idx, "attendance"] = results_2026.get(row["game_pk"])

played = df_2026[df_2026["status"] == "Final"]
upcoming = df_2026[df_2026["status"] != "Final"]
print("2026 home games:", len(df_2026))
print("  Already played:", len(played))
print("  Not yet played:", len(upcoming))
df_2026.head(10)


Fetching attendance for 46 completed 2026 games...
2026 home games: 81
  Already played: 46
  Not yet played: 35


,date,season,opponent,home_score,away_score,attendance,status,game_pk
0,2026-03-26,2026,Detroit Tigers,2.0,8.0,45673,Final,823325
1,2026-03-27,2026,Detroit Tigers,2.0,5.0,44896,Final,823324
2,2026-03-28,2026,Detroit Tigers,3.0,0.0,44368,Final,823323
3,2026-03-30,2026,San Francisco Giants,2.0,3.0,43611,Final,823320
4,2026-03-31,2026,San Francisco Giants,3.0,9.0,41891,Final,823322
5,2026-04-01,2026,San Francisco Giants,7.0,1.0,41491,Final,823321
6,2026-04-09,2026,Colorado Rockies,7.0,3.0,41390,Final,823319
7,2026-04-10,2026,Colorado Rockies,5.0,2.0,42454,Final,823318
8,2026-04-11,2026,Colorado Rockies,9.0,5.0,42318,Final,823316
9,2026-04-12,2026,Colorado Rockies,7.0,2.0,39786,Final,823317


In [31]:
df_2026.to_csv("../data/padres_2026_prediction.csv", index=False)
print("Saved to data/padres_2026_prediction.csv")

Saved to data/padres_2026_prediction.csv
